<a href="https://colab.research.google.com/github/claudiodanielpc/medium/blob/main/datos_abiertos_gobfed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
!pip install great_tables
from great_tables import GT

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/607.2 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.3 MB/s eta 0:00:00


In [3]:
headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36'}
##Url básica del portal de datos abiertos
url_basica="https://www.datos.gob.mx/api/action/"

In [4]:
#Nombre de los conjuntos de datos
url_datos=url_basica+"package_list"

nom_conjunto=requests.get(url_datos, headers=headers)
nom_conjunto=nom_conjunto.json()
nom_conjunto=nom_conjunto['result']
print("Total de temas:", len(nom_conjunto))

Total de temas: 954


In [5]:
##Obtener datos generales de cada tema
titulo = []
organizacion= []
org_corto=[]
categoria=[]
num_bases=[]
descripcion = []
frecuencia=[]
metadata_creado=[]
metadata_modificado=[]
url_ultimo_archivo=[]


#Buscar
for conjunto in nom_conjunto:
    url_datos=url_basica+"package_show?id="+conjunto
    r=requests.get(url_datos, headers=headers)
    r=r.json()
    titulo.append(r['result']['title'])
    if r['result'].get('description') is None:
        descripcion.append(None)
    else:
        descripcion.append(r['result']['description'])
    categoria.append(r['result']["groups"][0]["name"])
    num_bases.append(r['result']['num_resources'])
    organizacion.append(r['result']['organization']['title'])
    org_corto.append(r['result']['organization']['name'])
    recursos = r['result'].get('resources', [])
    if recursos:
      freq = recursos[0].get('update_frequency', None)
    else:
      freq = None
    frecuencia.append(freq)
    metadata_creado.append(r['result'].get('metadata_created'))
    metadata_modificado.append(r['result'].get('metadata_modified'))

In [6]:
##Tabla general
tabla_conjunto = pd.DataFrame({
    "tema": titulo,
    "organizacion": organizacion,
    "org_corto": org_corto,
    "categoria": categoria,
    "descripcion": descripcion,
    "frecuencia": frecuencia,
    "num_bases": num_bases,
    "metadata_creado": metadata_creado,
    "metadata_modificado": metadata_modificado,
})
tabla_conjunto

,tema,organizacion,org_corto,categoria,descripcion,frecuencia,num_bases,metadata_creado,metadata_modificado
0,Abasto de medicamentos y material de curación,Instituto de Seguridad y Servicios Sociales de...,issste,salud,None,Mensual,8,2025-07-23T15:37:32.481240,2025-09-24T15:11:29.711521
1,Académicos contratados en la Escuela,Instituto de Seguridad y Servicios Sociales de...,issste,educacion,None,Semestral,1,2025-07-18T18:42:19.846782,2025-07-18T19:00:32.606361
2,Accesibilidad a centros urbanos,Secretaría General del Consejo Nacional de Pob...,conapo,poblacion,None,,2,2025-03-16T21:38:39.975917,2025-06-04T19:42:48.644318
3,Accidentes de tránsito,"Secretaría de Infraestructura, Comunicaciones ...",secretaria_comunicaciones,movilidad,None,No aplica,2,2025-07-16T21:59:31.848894,2025-08-04T16:19:16.853973
4,Acciones de difusión de la ciencia y la tecnol...,Instituto Politécnico Nacional (IPN),ipn,educacion,None,Anual,1,2025-11-26T18:53:09.050886,2025-11-26T18:54:11.748185
...,...,...,...,...,...,...,...,...,...
949,Volumen de las exportaciones de petróleo crudo...,Petróleos Mexicanos (PEMEX),pemex,energia,None,Mensual,1,2025-09-18T23:32:11.231774,2025-09-25T18:40:49.421540
950,Volumen de las ventas internas de productos el...,Petróleos Mexicanos (PEMEX),pemex,energia,None,Mensual,1,2025-09-18T22:01:40.169524,2025-09-24T22:13:17.469263
951,Volumen de las ventas internas de productos pe...,Petróleos Mexicanos (PEMEX),pemex,energia,None,Mensual,1,2025-09-18T23:28:08.114257,2025-09-25T18:28:15.580843
952,Zonas Arqueológicas abiertas al público,Instituto Nacional de Antropología e Historia ...,inah,cultura,None,Trimestral,1,2025-11-26T17:56:40.327500,2025-11-26T17:57:51.983663


In [7]:
#Contar número de conjuntos de datos
print("Total de conjuntos de datos:", sum(tabla_conjunto['num_bases']))

Total de conjuntos de datos: 4708


In [13]:
from great_tables import GT

def tabla_frecuencias_gt(
    df,
    var,
    titulo,
    top_n=None,
    bottom_n=None
):

    if top_n is not None and bottom_n is not None:
        raise ValueError("Usa solo uno: top_n o bottom_n")

    tabla = (
        df[var]
            .value_counts()
            .to_frame(name="Número")
            .assign(
                pct=(
                    df[var]
                        .value_counts(normalize=True)
                        .mul(100)
                        .round(1)
                )
            )
            .sort_values("Número", ascending=False)
            .reset_index(names=var)
    )

    if top_n is not None:
        tabla = tabla.head(top_n)

    if bottom_n is not None:
        tabla = tabla.tail(bottom_n)

    return (
        GT(tabla)
            .tab_header(
                title=titulo,
                subtitle="Número y porcentaje del total"
            )
            .fmt_number(
                columns="Número",
                decimals=0,
                use_seps=True
            )
            .fmt_number(
                columns="pct",
                decimals=1
            )
            .cols_label(
                {
                    var: var.capitalize(),
                    "pct": "Porcentaje (%)"
                }
            )
            .cols_align(
                align="center",
                columns=["Número", "pct"]
            )
            .cols_align(
                align="left",
                columns=var
            )
    )


In [14]:
##Número de instituciones con datos en el portal
tabla_conjunto["organizacion"].nunique()

133

In [15]:
## Temas por institución
tabla_frecuencias_gt(
    tabla_conjunto,
    var="organizacion",
    titulo="Top 30 instituciones con más aporte de datos",
    top_n=30
)

GT(_tbl_data=                                         organizacion  Número  pct
0   Servicio Nacional de Sanidad, Inocuidad y Cali...      74  7.8
1                Instituto Politécnico Nacional (IPN)      46  4.8
2   Comisión Nacional de Acuacultura y Pesca (CONA...      39  4.1
3                         Petróleos Mexicanos (PEMEX)      31  3.2
4   Instituto de Seguridad y Servicios Sociales de...      29  3.0
5           Secretaría de Relaciones Exteriores (SRE)      26  2.7
6   Coordinación Nacional de Becas para el Bienest...      21  2.2
7       Comisión Nacional de Seguros y Fianzas (CNSF)      19  2.0
8   Instituto Nacional de Antropología e Historia ...      19  2.0
9   Impresora y Encuadernadora Progreso, S.A de C....      18  1.9
10  Centro de Investigación en Alimentación y Desa...      18  1.9
11   Secretaría Anticorrupción y Buen Gobierno (SABG)      18  1.9
12  Instituto Nacional de Psiquiatría "Ramón de la...      18  1.9
13  Centro de Investigaciones Biológicas del Noroe...      15  1.6
14   Secretaría del Trabajo y Previsión Social (STPS)      15  1.6
15  Secretaría de Agricultura y Desarrollo Rural (...      15  1.6
16                 Servicio Postal Mexicano (SEPOMEX)      14  1.5
17  Instituto para la Protección al Ahorro Bancari...      14  1.5
18              Prevención y Reinserción Social (PRS)      12  1.3
19  Instituto Nacional de Estudios Históricos de l...      12  1.3
20  Caminos y Puentes Federales de Ingresos y Serv...      12  1.3
21  Colegio Nacional de Educación Profesional Técn...      11  1.2
22  Comisión Nacional del Sistema de Ahorro para e...      11  1.2
23             Secretaría de la Función Pública (SFP)      11  1.2
24  Secretaría General del Consejo Nacional de Pob...      10  1.0
25                              Secretaría de Cultura      10  1.0
26  Instituto Nacional de Bellas Artes y Literatur...      10  1.0
27  Instituto de Seguridad Social para las Fuerzas...      10  1.0
28  Sistema Público de Radiodifusión del Estado Me...      10  1.0
29                 Comisión Nacional de Energía (CNE)       9  0.9, _body=<great_tables._gt_data.Body object at 0x7cbccf066c00>, _boxhead=Boxhead([ColInfo(var='organizacion', type=<ColInfoTypeEnum.default: 1>, column_label='Organizacion', column_align='left', column_width=None), ColInfo(var='Número', type=<ColInfoTypeEnum.default: 1>, column_label='Número', column_align='center', column_width=None), ColInfo(var='pct', type=<ColInfoTypeEnum.default: 1>, column_label='Porcentaje (%)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7cbce9adb6e0>, _spanners=Spanners([]), _heading=Heading(title='Top 30 instituciones con más aporte de datos', subtitle='Número y porcentaje del total', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7cbcce71d130>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7cbcce71dd30>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7cbcce71c1a0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7cbcce71c0b0>, <great_tables._gt_data.FormatInfo object at 0x7cbcce71c6b0>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system'

In [23]:
##Temas por institución. Las que menos participan
tabla_frecuencias_gt(
    tabla_conjunto,
    var="organizacion",
    titulo="Instituciones con menos aporte de datos",
    bottom_n=24
)

GT(_tbl_data=                                          organizacion  Número  pct
109          Seguridad Alimentaria Mexicana (SEGALMEX)       1  0.1
110  Instituto Nacional de Ecología y Cambio Climát...       1  0.1
111  Administración del Sistema Portuario Nacional ...       1  0.1
112                  Secretaría de Gobernación (SEGOB)       1  0.1
113               Comisión Nacional del Agua (CONAGUA)       1  0.1
114    Secretaría de Hacienda y Crédito Público (SHCP)       1  0.1
115     Consejo Nacional de Fomento Educativo (CONAFE)       1  0.1
116              Instituto Mexicano de la Radio (IMER)       1  0.1
117                    Registro Agrario Nacional (RAN)       1  0.1
118  Secretariado Ejecutivo del Sistema Nacional de...       1  0.1
119               Comisión Reguladora de Energía (CRE)       1  0.1
120    Instituto Nacional de Lenguas Indígenas (INALI)       1  0.1
121  Consejo Nacional para Prevenir la Discriminaci...       1  0.1
122                            Secretaría de Bienestar       1  0.1
123  Instituto de Administración y Avalúos de Biene...       1  0.1
124                     Secretaría de Turismo (SECTUR)       1  0.1
125      Instituto Mexicano de Cinematografía (IMCINE)       1  0.1
126  Secretaría de Ciencia, Humanidades, Tecnología...       1  0.1
127  Instituto Nacional para el Federalismo y el De...       1  0.1
128              Exportadora de Sal, S.A de C.V (ESSA)       1  0.1
129         Instituto Nacional de Cancerología (INCAN)       1  0.1
130  Instituto Nacional de Investigaciones Nucleare...       1  0.1
131             Comisión Nacional de Vivienda (CONAVI)       1  0.1
132  Agencia de Transformación Digital y Telecomuni...       1  0.1, _body=<great_tables._gt_data.Body object at 0x7cbcce774740>, _boxhead=Boxhead([ColInfo(var='organizacion', type=<ColInfoTypeEnum.default: 1>, column_label='Organizacion', column_align='left', column_width=None), ColInfo(var='Número', type=<ColInfoTypeEnum.default: 1>, column_label='Número', column_align='center', column_width=None), ColInfo(var='pct', type=<ColInfoTypeEnum.default: 1>, column_label='Porcentaje (%)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7cbcce756990>, _spanners=Spanners([]), _heading=Heading(title='Instituciones con menos aporte de datos', subtitle='Número y porcentaje del total', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7cbcce774d40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7cbcce774d10>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7cbcce774ce0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7cbcce774230>, <great_tables._gt_data.FormatInfo object at 0x7cbcce774b60>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', ty

In [11]:
###Temas por categoría
tabla_frecuencias_gt(
    tabla_conjunto,
    var="frecuencia",
    titulo="Temas por frecuencia de actualización"
)

GT(_tbl_data=   frecuencia  Número   pct
0       Anual     360  37.7
1  Trimestral     202  21.2
2     Mensual     186  19.5
3   Semestral      85   8.9
4                  56   5.9
5   No aplica      48   5.0
6   Bimestral      10   1.0
7   Histórico       5   0.5
8     Semanal       2   0.2, _body=<great_tables._gt_data.Body object at 0x7cbccf072540>, _boxhead=Boxhead([ColInfo(var='frecuencia', type=<ColInfoTypeEnum.default: 1>, column_label='Frecuencia', column_align='left', column_width=None), ColInfo(var='Número', type=<ColInfoTypeEnum.default: 1>, column_label='Número', column_align='center', column_width=None), ColInfo(var='pct', type=<ColInfoTypeEnum.default: 1>, column_label='Porcentaje (%)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7cbccf01b8f0>, _spanners=Spanners([]), _heading=Heading(title='Temas por frecuencia de actualización', subtitle='Número y porcentaje del total', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7cbccf073740>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7cbccf073470>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7cbccf073440>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7cbccf0734d0>, <great_tables._gt_data.FormatInfo object at 0x7cbccf073350>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='

In [12]:
## Temas por categoría
tabla_frecuencias_gt(
    tabla_conjunto,
    var="categoria",
    titulo="Temas por categoría"
)

GT(_tbl_data=             categoria  Número   pct
0            educacion     120  12.6
1              cultura      80   8.4
2          agricultura      78   8.2
3             economia      76   8.0
4                salud      70   7.3
5              trabajo      64   6.7
6             gobierno      56   5.9
7   ciencia_tecnologia      50   5.2
8              energia      39   4.1
9          presupuesto      38   4.0
10           servicios      37   3.9
11     infraestructura      35   3.7
12           mar_costa      34   3.6
13  programas_sociales      29   3.0
14           seguridad      25   2.6
15      medio_ambiente      24   2.5
16  telecomunicaciones      19   2.0
17           migracion      15   1.6
18    derechos_humanos      15   1.6
19          territorio      13   1.4
20           poblacion      11   1.2
21           movilidad       9   0.9
22             mujeres       7   0.7
23             deporte       5   0.5
24             turismo       4   0.4
25   multiculturalidad       1   0.1, _body=<great_tables._gt_data.Body object at 0x7cbccf072600>, _boxhead=Boxhead([ColInfo(var='categoria', type=<ColInfoTypeEnum.default: 1>, column_label='Categoria', column_align='left', column_width=None), ColInfo(var='Número', type=<ColInfoTypeEnum.default: 1>, column_label='Número', column_align='center', column_width=None), ColInfo(var='pct', type=<ColInfoTypeEnum.default: 1>, column_label='Porcentaje (%)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7cbccf05c7a0>, _spanners=Spanners([]), _heading=Heading(title='Temas por categoría', subtitle='Número y porcentaje del total', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7cbccf0706b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7cbccf073c50>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7cbccf070350>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7cbccf070890>, <great_tables._gt_data.FormatInfo object at 0x7cbccf070050>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=